# WiDS Global Datathon 2026 — Survival Submit

In [1]:
import os, sys, json, math, warnings, subprocess, re, hashlib
from io import StringIO
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" or "fast"
DO_OOF = True
AUTO_INSTALL_SKSURV = True
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    for root, dirs, files in os.walk("/kaggle/input"):
        files = set(files)
        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
            return root
    raise FileNotFoundError("Could not find competition data directory.")

DATA_DIR = find_data_dir()

def ensure_pkg(pkg, import_name=None, allow_install=True):
    name = import_name or pkg
    try:
        __import__(name)
        return True
    except Exception:
        if not allow_install:
            return False
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            __import__(name)
            return True
        except Exception as e:
            print(f"[WARN] package unavailable: {pkg} -> {e}")
            return False

HAS_SKSURV = ensure_pkg("scikit-survival", "sksurv", AUTO_INSTALL_SKSURV)
ensure_pkg("lightgbm", "lightgbm", True)
ensure_pkg("catboost", "catboost", True)

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from catboost import CatBoostClassifier

if HAS_SKSURV:
    from sksurv.util import Surv
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
    from sksurv.linear_model import CoxPHSurvivalAnalysis

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape, "HAS_SKSURV =", HAS_SKSURV)

def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1)
    speed = r["closing_speed_m_per_h"]
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"]
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0)
    align = r["alignment_abs"]
    fire_radius = np.sqrt(np.maximum(area_first, 0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0)
    eff_close = closing_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (dist / 1000.0 + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius / 1000.0
    r["radius_to_dist"] = fire_radius / dist
    r["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])

    r["threat_score"] = align * speed / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * speed
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = speed * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0)

    r["zone_3km"] = (dist < 3000).astype(float)
    r["zone_near"] = (dist < 5000).astype(float)
    r["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    r["zone_far"] = (dist >= 10000).astype(float)
    r["zone_20km"] = (dist < 20000).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1).values
    ref_eff = (ref["closing_speed_m_per_h"].clip(lower=0).values +
               ref["radial_growth_rate_m_per_h"].clip(lower=0).values)
    ref_threat = (
        ref["alignment_abs"].values * ref_eff /
        np.log1p(ref["dist_min_ci_0_5h"].clip(lower=1).values)
    )

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref["dist_min_ci_0_5h"].clip(lower=1).values < 5000
    cur_near = dist.values < 5000
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0)
    return r

train_proc = create_features(train_df, fit_df=train_df)
test_proc = create_features(test_df, fit_df=train_df)

time_values = train_df["time_to_hit_hours"].values
event_values = train_df["event"].astype(int).values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values

def compute_c_index(time, event, risk):
    n = len(time)
    conc = 0.0
    comp = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1
            if risk[i] > risk[j]:
                conc += 1
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_brier(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    prob = np.clip(prob[valid], 0, 1)
    return float(np.mean((prob - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb), c_idx, wb, b24, b48, b72

def enforce_monotonicity(preds):
    out = np.clip(preds, 0, 1)
    for i in range(1, out.shape[1]):
        out[:, i] = np.maximum(out[:, i], out[:, i - 1])
    return out

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0
    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights

def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p

def soft_gate(dist_m, center, width):
    z = np.clip((dist_m - center) / max(width, 1.0), -60, 60)
    return 1.0 / (1.0 + np.exp(z))

def stabilize(pred, anchor, alpha):
    return np.clip(alpha * pred + (1.0 - alpha) * anchor, 0, 1)

def repeated_isotonic(train_signal, test_signal, y, mask, seeds, n_splits=5):
    train_signal = np.asarray(train_signal, dtype=float)
    test_signal = np.asarray(test_signal, dtype=float)
    valid_idx = np.where(mask)[0]
    yv = y[valid_idx]
    oof = np.zeros(len(train_signal), dtype=float)
    cnt = np.zeros(len(train_signal), dtype=float)
    full_fill = np.zeros(len(train_signal), dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            ir.fit(train_signal[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(train_signal[va_idx])
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        ir_full.fit(train_signal[valid_idx], yv)
        full_fill += ir_full.predict(train_signal)
        test_pred += ir_full.predict(test_signal)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

if HAS_SKSURV:
    HORIZONS = [12, 24, 48, 72]

    def get_surv_predictions(model, X):
        surv_fns = model.predict_survival_function(X)
        preds = np.empty((len(surv_fns), len(HORIZONS)), dtype=float)
        for i, fn in enumerate(surv_fns):
            try:
                t_min, t_max = fn.domain
            except Exception:
                t_min, t_max = fn.x[0], fn.x[-1]
            grid = np.clip(HORIZONS, t_min, t_max)
            preds[i, :] = 1.0 - fn(grid)
        return np.clip(preds, 0, 1)

    y_surv = Surv.from_arrays(
        event=train_df["event"].astype(bool).values,
        time=train_df["time_to_hit_hours"].values
    )

    def repeated_survival_model(X_train, X_test, build_model, seeds, n_splits=5):
        n_train = len(X_train)
        n_test = len(X_test)
        oof = np.zeros((n_train, 4), dtype=float)
        cnt = np.zeros(n_train, dtype=float)
        full_fill = np.zeros((n_train, 4), dtype=float)
        test_pred = np.zeros((n_test, 4), dtype=float)

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            seed_test = np.zeros((n_test, 4), dtype=float)

            for tr_idx, va_idx in cv.split(X_train, event_values):
                model = build_model(seed)
                try:
                    model.fit(X_train.iloc[tr_idx], y_surv[tr_idx])
                    oof[va_idx] += get_surv_predictions(model, X_train.iloc[va_idx])
                    cnt[va_idx] += 1.0
                    seed_test += get_surv_predictions(model, X_test) / n_splits
                except Exception:
                    oof[va_idx] += 0.5
                    cnt[va_idx] += 1.0
                    seed_test += 0.5 / n_splits

            test_pred += seed_test / len(seeds)

            model_full = build_model(seed)
            try:
                model_full.fit(X_train, y_surv)
                full_fill += get_surv_predictions(model_full, X_train)
            except Exception:
                full_fill += 0.5

        full_fill /= len(seeds)
        nz = cnt > 0
        oof[nz] /= cnt[nz, None]
        oof[~nz] = full_fill[~nz]
        return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

FAST_SEEDS = (123, 456, 789)
BASE_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033, 279, 239, 70, 77, 31, 2024)
GBSA_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 4640, 841, 7755, 8525, 2701, 8817)
COX_SEEDS_FULL = BASE_SEEDS[:12]
RSF_SEEDS_FULL = BASE_SEEDS[:10]
TREE_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 123456, 654321)
CAT_SEEDS_FULL = BASE_SEEDS[:8]

def choose_seeds(full_seeds):
    return FAST_SEEDS if RUN_MODE == "fast" else tuple(full_seeds)

RAW_FEATURES = [c for c in train_df.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

COX_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
COX_FEATURES = [c for c in COX_FEATURES if c in train_proc.columns]

NEAR_LGB_FEATURES = [
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
    "eta_effective", "log_eta_effective", "dist_km", "threat_score_eff",
    "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
    "area_first_ha", "fire_radius_km", "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
]
NEAR_LGB_FEATURES = [c for c in NEAR_LGB_FEATURES if c in train_proc.columns]

FAR_LGB_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "alignment_abs",
    "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "num_perimeters_0_5h",
    "far_threat_rank", "is_summer", "zone_far",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "log1p_growth", "dist_fit_r2_0_5h",
]
FAR_LGB_FEATURES = [c for c in FAR_LGB_FEATURES if c in train_proc.columns]

GLOBAL_TREE_FEATURES = sorted(set(
    NEAR_LGB_FEATURES + FAR_LGB_FEATURES + [
        "inv_distance_sq", "sqrt_distance", "threat_score", "growth_intensity",
        "speed_alignment", "effective_alignment", "zone_warning",
        "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
        "projected_advance_pos", "closing_distance_pos", "slope_toward",
        "log_area_dist_ratio", "fire_radius_km", "log1p_area_first",
        "dist_rank_global", "eff_rank_global", "threat_rank_global",
    ]
))
GLOBAL_TREE_FEATURES = [c for c in GLOBAL_TREE_FEATURES if c in train_proc.columns]

LINEAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

X_raw_train = train_df[RAW_FEATURES].copy()
X_raw_test = test_df[RAW_FEATURES].copy()

if HAS_SKSURV:
    coxt = train_proc[COX_FEATURES].copy()
    non_const = coxt.std(axis=0) > 0
    COX_FEATURES = list(coxt.columns[non_const.values])
    scaler_cox = StandardScaler()
    X_cox_train = pd.DataFrame(
        scaler_cox.fit_transform(train_proc[COX_FEATURES]),
        columns=COX_FEATURES, index=train_proc.index
    )
    X_cox_test = pd.DataFrame(
        scaler_cox.transform(test_proc[COX_FEATURES]),
        columns=COX_FEATURES, index=test_proc.index
    )

X_near_lgb_train = train_proc[NEAR_LGB_FEATURES].copy()
X_near_lgb_test = test_proc[NEAR_LGB_FEATURES].copy()
X_far_lgb_train = train_proc[FAR_LGB_FEATURES].copy()
X_far_lgb_test = test_proc[FAR_LGB_FEATURES].copy()
X_global_train = train_proc[GLOBAL_TREE_FEATURES].copy()
X_global_test = test_proc[GLOBAL_TREE_FEATURES].copy()
X_linear_train = train_proc[LINEAR_FEATURES].copy()
X_linear_test = test_proc[LINEAR_FEATURES].copy()

def make_lgb_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(objective="binary", random_state=seed, n_jobs=-1, verbose=-1))
        return lgb.LGBMClassifier(**p)
    return _build

def make_cat_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        ))
        return CatBoostClassifier(**p)
    return _build

def make_logit_builder(C):
    def _build(seed):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=C, penalty="l2", solver="lbfgs", max_iter=4000)),
        ])
    return _build

gbsa_configs = [
    {"learning_rate": 0.01,  "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.005, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000},
    {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400},
    {"learning_rate": 0.008, "subsample": 0.75, "max_depth": 2, "min_samples_leaf": 15, "min_samples_split": 4, "n_estimators": 1500},
    {"learning_rate": 0.015, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 1000},
    {"learning_rate": 0.005, "subsample": 0.90, "max_depth": 3, "min_samples_leaf": 18, "min_samples_split": 5, "n_estimators": 2500},
    {"learning_rate": 0.01,  "subsample": 0.80, "max_depth": 4, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
]
cox_alphas = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
rsf_configs = [
    {"n_estimators": 200, "min_samples_leaf": 12, "max_features": "sqrt", "max_depth": None},
    {"n_estimators": 200, "min_samples_leaf": 18, "max_features": "sqrt", "max_depth": None},
    {"n_estimators": 200, "min_samples_leaf": 12, "max_features": 0.5,    "max_depth": 5},
]

if RUN_MODE == "fast":
    gbsa_configs = gbsa_configs[:3]
    cox_alphas = [0.01, 0.10, 1.00]
    rsf_configs = rsf_configs[:1]

global_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.035, "n_estimators": 260, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.8, "reg_lambda": 2.5, "num_leaves": 4},
    48: {"max_depth": 3, "learning_rate": 0.030, "n_estimators": 260, "subsample": 0.85, "colsample_bytree": 0.85,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 6},
}

global_cat_cfgs = {
    12: {"iterations": 300, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 10.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    24: {"iterations": 320, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    48: {"iterations": 340, "learning_rate": 0.028, "depth": 4, "l2_leaf_reg": 14.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
}

near_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 3, "reg_alpha": 0.3, "reg_lambda": 1.0, "num_leaves": 4},
}

far_lgb_cfgs = {
    24: {"max_depth": 2, "learning_rate": 0.030, "n_estimators": 200, "subsample": 0.70, "colsample_bytree": 0.70,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 4},
}

linear_cs = {12: 0.60, 24: 0.80, 48: 1.00}

y_map = {}
mask_map = {}
for h in [12, 24, 48]:
    y_map[h], mask_map[h] = make_binary_target(time_values, event_values, h)

ANCHOR_SEEDS = choose_seeds(BASE_SEEDS[:8])
TREE_SEEDS = choose_seeds(TREE_SEEDS_FULL)
CAT_SEEDS = choose_seeds(CAT_SEEDS_FULL)
LINEAR_SEEDS = choose_seeds(BASE_SEEDS[:8])

anchor_signal_train = 1.0 / (train_proc["dist_km"].values + 0.25)
anchor_signal_test = 1.0 / (test_proc["dist_km"].values + 0.25)

anchor_oof, anchor_test = {}, {}
for h in [12, 24, 48]:
    anchor_oof[h], anchor_test[h] = repeated_isotonic(
        anchor_signal_train, anchor_signal_test, y_map[h], mask_map[h], ANCHOR_SEEDS, n_splits=5
    )
    print(f"anchor@{h}  Brier = {compute_brier(time_values, event_values, anchor_oof[h], h):.6f}")

global_lgb_oof, global_lgb_test = {}, {}
global_cat_oof, global_cat_test = {}, {}
linear_oof, linear_test = {}, {}

for h in [12, 24, 48]:
    global_lgb_oof[h], global_lgb_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_lgb_builder(global_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_lgb_oof[h] = stabilize(global_lgb_oof[h], anchor_oof[h], 0.92)
    global_lgb_test[h] = stabilize(global_lgb_test[h], anchor_test[h], 0.92)
    print(f"global_lgb@{h}  Brier = {compute_brier(time_values, event_values, global_lgb_oof[h], h):.6f}")

    global_cat_oof[h], global_cat_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_cat_builder(global_cat_cfgs[h]), CAT_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_cat_oof[h] = stabilize(global_cat_oof[h], anchor_oof[h], 0.90)
    global_cat_test[h] = stabilize(global_cat_test[h], anchor_test[h], 0.90)
    print(f"global_cat@{h}  Brier = {compute_brier(time_values, event_values, global_cat_oof[h], h):.6f}")

    linear_oof[h], linear_test[h] = repeated_binary_model(
        X_linear_train, X_linear_test, y_map[h], mask_map[h],
        make_logit_builder(linear_cs[h]), LINEAR_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    linear_oof[h] = stabilize(linear_oof[h], anchor_oof[h], 0.88)
    linear_test[h] = stabilize(linear_test[h], anchor_test[h], 0.88)
    print(f"linear@{h}      Brier = {compute_brier(time_values, event_values, linear_oof[h], h):.6f}")

near_lgb_oof, near_lgb_test = {}, {}
for h in [12, 24, 48]:
    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model(
        X_near_lgb_train, X_near_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(near_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    near_lgb_oof[h] = stabilize(near_lgb_oof[h], anchor_oof[h], 0.94)
    near_lgb_test[h] = stabilize(near_lgb_test[h], anchor_test[h], 0.94)
    print(f"near_lgb@{h}    Brier = {compute_brier(time_values, event_values, near_lgb_oof[h], h):.6f}")

far_lgb_oof, far_lgb_test = {}, {}
for h in [24, 48]:
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model(
        X_far_lgb_train, X_far_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(far_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    far_lgb_oof[h] = stabilize(far_lgb_oof[h], anchor_oof[h], 0.94)
    far_lgb_test[h] = stabilize(far_lgb_test[h], anchor_test[h], 0.94)
    print(f"far_lgb@{h}     Brier = {compute_brier(time_values, event_values, far_lgb_oof[h], h):.6f}")

timing_signal_train = 1.0 / (np.asarray(train_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_signal_test = 1.0 / (np.asarray(test_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_oof, timing_test = {}, {}
for h in [12, 24, 48]:
    timing_mask = mask_map[h] & ((dist_train < 8000) | (y_map[h] == 1))
    timing_oof[h], timing_test[h] = repeated_isotonic(
        timing_signal_train, timing_signal_test, y_map[h], timing_mask, ANCHOR_SEEDS, n_splits=5
    )
    timing_oof[h] = stabilize(timing_oof[h], anchor_oof[h], 0.90)
    timing_test[h] = stabilize(timing_test[h], anchor_test[h], 0.90)
    print(f"timing@{h}      Brier = {compute_brier(time_values, event_values, timing_oof[h], h):.6f}")

base_oof = {
    "anchor": anchor_oof,
    "glob_lgb": global_lgb_oof,
    "glob_cat": global_cat_oof,
    "lin": linear_oof,
    "timing": timing_oof,
}
base_test = {
    "anchor": anchor_test,
    "glob_lgb": global_lgb_test,
    "glob_cat": global_cat_test,
    "lin": linear_test,
    "timing": timing_test,
}

if HAS_SKSURV:
    GBSA_SEEDS = choose_seeds(GBSA_SEEDS_FULL)
    COX_SEEDS = choose_seeds(COX_SEEDS_FULL)
    RSF_SEEDS = choose_seeds(RSF_SEEDS_FULL)

    gbsa_oof = np.zeros((len(train_df), 4), dtype=float)
    gbsa_test = np.zeros((len(test_df), 4), dtype=float)
    for cfg in gbsa_configs:
        oof_cfg, test_cfg = repeated_survival_model(
            X_raw_train, X_raw_test,
            lambda seed, cfg=cfg: GradientBoostingSurvivalAnalysis(random_state=seed, **cfg),
            GBSA_SEEDS, n_splits=5
        )
        gbsa_oof += oof_cfg / len(gbsa_configs)
        gbsa_test += test_cfg / len(gbsa_configs)

    gbsa_oof[:, 1] = np.clip(gbsa_oof[:, 1] ** 1.08, 0, 1)
    gbsa_test[:, 1] = np.clip(gbsa_test[:, 1] ** 1.08, 0, 1)
    for i, h in enumerate([12, 24, 48]):
        gbsa_oof[:, i] = stabilize(gbsa_oof[:, i], anchor_oof[h], 0.98 if h != 24 else 0.97)
        gbsa_test[:, i] = stabilize(gbsa_test[:, i], anchor_test[h], 0.98 if h != 24 else 0.97)
        print(f"gbsa@{h}        Brier = {compute_brier(time_values, event_values, gbsa_oof[:, i], h):.6f}")

    cox_oof = np.zeros((len(train_df), 4), dtype=float)
    cox_test = np.zeros((len(test_df), 4), dtype=float)
    for alpha in cox_alphas:
        oof_alpha, test_alpha = repeated_survival_model(
            X_cox_train, X_cox_test,
            lambda seed, a=alpha: CoxPHSurvivalAnalysis(alpha=a),
            COX_SEEDS, n_splits=5
        )
        cox_oof += oof_alpha / len(cox_alphas)
        cox_test += test_alpha / len(cox_alphas)
    for i, h in enumerate([12, 24, 48]):
        cox_oof[:, i] = stabilize(cox_oof[:, i], anchor_oof[h], 0.97)
        cox_test[:, i] = stabilize(cox_test[:, i], anchor_test[h], 0.97)
        print(f"cox@{h}         Brier = {compute_brier(time_values, event_values, cox_oof[:, i], h):.6f}")

    rsf_oof = np.zeros((len(train_df), 4), dtype=float)
    rsf_test = np.zeros((len(test_df), 4), dtype=float)
    for cfg in rsf_configs:
        oof_cfg, test_cfg = repeated_survival_model(
            X_raw_train, X_raw_test,
            lambda seed, cfg=cfg: RandomSurvivalForest(random_state=seed, n_jobs=-1, **cfg),
            RSF_SEEDS, n_splits=5
        )
        rsf_oof += oof_cfg / len(rsf_configs)
        rsf_test += test_cfg / len(rsf_configs)
    for i, h in enumerate([12, 24, 48]):
        rsf_oof[:, i] = stabilize(rsf_oof[:, i], anchor_oof[h], 0.90)
        rsf_test[:, i] = stabilize(rsf_test[:, i], anchor_test[h], 0.90)
        print(f"rsf@{h}         Brier = {compute_brier(time_values, event_values, rsf_oof[:, i], h):.6f}")

    base_oof["gbsa"] = {12: gbsa_oof[:, 0], 24: gbsa_oof[:, 1], 48: gbsa_oof[:, 2]}
    base_oof["cox"] = {12: cox_oof[:, 0], 24: cox_oof[:, 1], 48: cox_oof[:, 2]}
    base_oof["rsf"] = {12: rsf_oof[:, 0], 24: rsf_oof[:, 1], 48: rsf_oof[:, 2]}

    base_test["gbsa"] = {12: gbsa_test[:, 0], 24: gbsa_test[:, 1], 48: gbsa_test[:, 2]}
    base_test["cox"] = {12: cox_test[:, 0], 24: cox_test[:, 1], 48: cox_test[:, 2]}
    base_test["rsf"] = {12: rsf_test[:, 0], 24: rsf_test[:, 1], 48: rsf_test[:, 2]}
else:
    print("[WARN] scikit-survival unavailable: using classifier stack + isotonic anchor only.")

near_lgb_store = {"oof": near_lgb_oof, "test": near_lgb_test}
far_lgb_store = {"oof": far_lgb_oof, "test": far_lgb_test}

PRIORS = {
    (12, "near"): {"gbsa": 0.60, "cox": 0.10, "rsf": 0.02, "zone_lgb": 0.10, "timing": 0.06, "glob_lgb": 0.05, "glob_cat": 0.03, "lin": 0.02, "anchor": 0.02},
    (12, "far"):  {"gbsa": 0.78, "glob_lgb": 0.08, "glob_cat": 0.05, "lin": 0.04, "anchor": 0.05},
    (24, "near"): {"gbsa": 0.66, "cox": 0.12, "rsf": 0.02, "zone_lgb": 0.05, "timing": 0.05, "glob_lgb": 0.04, "glob_cat": 0.03, "lin": 0.02, "anchor": 0.01},
    (24, "far"):  {"gbsa": 0.52, "cox": 0.20, "rsf": 0.05, "zone_lgb": 0.10, "glob_lgb": 0.06, "glob_cat": 0.04, "lin": 0.02, "anchor": 0.01},
    (48, "near"): {"gbsa": 0.54, "cox": 0.14, "rsf": 0.03, "zone_lgb": 0.09, "timing": 0.06, "glob_lgb": 0.06, "glob_cat": 0.04, "lin": 0.02, "anchor": 0.02},
    (48, "far"):  {"gbsa": 0.26, "cox": 0.18, "rsf": 0.06, "zone_lgb": 0.22, "glob_lgb": 0.14, "glob_cat": 0.08, "lin": 0.03, "anchor": 0.03},
}
BLEND_CFG = {
    (12, "near"): {"shrink": 0.25, "reg": 0.12},
    (12, "far"):  {"shrink": 0.35, "reg": 0.10},
    (24, "near"): {"shrink": 0.30, "reg": 0.10},
    (24, "far"):  {"shrink": 0.45, "reg": 0.08},
    (48, "near"): {"shrink": 0.35, "reg": 0.08},
    (48, "far"):  {"shrink": 0.50, "reg": 0.06},
}
GATE_CFG = {
    12: (4500.0, 1200.0),
    24: (5000.0, 1500.0),
    48: (6000.0, 1800.0),
}

def prior_to_vector(prior_dict, names):
    w = np.array([prior_dict.get(n, 0.0) for n in names], dtype=float)
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.ones(len(names), dtype=float)
    return w / w.sum()

def assemble_available(h, zone, split_kind, prior_dict):
    store = base_oof if split_kind == "oof" else base_test
    avail = {}
    for key in ["gbsa", "cox", "rsf", "glob_lgb", "glob_cat", "zone_lgb", "lin", "anchor", "timing"]:
        if key == "zone_lgb":
            src = near_lgb_store[split_kind] if zone == "near" else far_lgb_store[split_kind]
            if h in src:
                avail[key] = src[h]
        else:
            if key in store and h in store[key]:
                avail[key] = store[key][h]
    names = [n for n in prior_dict if n in avail]
    if not names:
        names = list(avail.keys())
    P = np.column_stack([avail[n] for n in names])
    prior = prior_to_vector(prior_dict, names)
    return names, P, prior

def optimize_blend(P, y, prior, reg, shrink):
    if P.shape[1] == 1:
        return prior.copy(), prior.copy()
    P = np.clip(P, 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)

    def objective(w):
        pred = P @ w
        return np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * P.shape[1]
    try:
        res = minimize(
            objective, prior.copy(), method="SLSQP", bounds=bounds, constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10}
        )
        if res.success:
            opt = np.clip(res.x, 0, None)
            opt = opt / opt.sum()
        else:
            opt = prior.copy()
    except Exception:
        opt = prior.copy()

    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 0, None)
    final = final / final.sum()
    return final, opt

weights_log = {}
oof_final = np.zeros((len(train_df), 4), dtype=float)
test_final = np.zeros((len(test_df), 4), dtype=float)

for col_idx, h in enumerate([12, 24, 48]):
    center, width = GATE_CFG[h]
    gate_train = soft_gate(dist_train, center, width)
    gate_test = soft_gate(dist_test, center, width)

    prior_near = PRIORS[(h, "near")]
    names_n, Pn_oof, prior_vec_n = assemble_available(h, "near", "oof", prior_near)
    _, Pn_test, _ = assemble_available(h, "near", "test", prior_near)
    near_mask = mask_map[h] & (dist_train < 5000)
    if near_mask.sum() >= max(8, len(names_n) + 2):
        w_n, opt_n = optimize_blend(
            Pn_oof[near_mask], y_map[h][near_mask], prior_vec_n,
            BLEND_CFG[(h, "near")]["reg"], BLEND_CFG[(h, "near")]["shrink"]
        )
    else:
        w_n, opt_n = prior_vec_n.copy(), prior_vec_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0, 1)
    pred_near_test = np.clip(Pn_test @ w_n, 0, 1)

    prior_far = PRIORS[(h, "far")]
    names_f, Pf_oof, prior_vec_f = assemble_available(h, "far", "oof", prior_far)
    _, Pf_test, _ = assemble_available(h, "far", "test", prior_far)
    far_mask = mask_map[h] & ~(dist_train < 5000)
    if far_mask.sum() >= max(8, len(names_f) + 2):
        w_f, opt_f = optimize_blend(
            Pf_oof[far_mask], y_map[h][far_mask], prior_vec_f,
            BLEND_CFG[(h, "far")]["reg"], BLEND_CFG[(h, "far")]["shrink"]
        )
    else:
        w_f, opt_f = prior_vec_f.copy(), prior_vec_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0, 1)
    pred_far_test = np.clip(Pf_test @ w_f, 0, 1)

    oof_final[:, col_idx] = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
    test_final[:, col_idx] = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    weights_log[f"{h}_near"] = {
        "names": names_n,
        "prior": [float(x) for x in prior_vec_n],
        "opt": [float(x) for x in opt_n],
        "final": [float(x) for x in w_n],
    }
    weights_log[f"{h}_far"] = {
        "names": names_f,
        "prior": [float(x) for x in prior_vec_f],
        "opt": [float(x) for x in opt_f],
        "final": [float(x) for x in w_f],
    }

oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)

exp_oof_final = oof_final.copy()
exp_test_final = test_final.copy()

if HAS_SKSURV:
    gbsa_core_oof = np.column_stack([
        base_oof["gbsa"][12],
        np.clip(base_oof["gbsa"][24] ** 1.10, 0, 1),
        base_oof["gbsa"][48],
        np.ones(len(train_df), dtype=float),
    ])
    gbsa_core_test = np.column_stack([
        base_test["gbsa"][12],
        np.clip(base_test["gbsa"][24] ** 1.10, 0, 1),
        base_test["gbsa"][48],
        np.ones(len(test_df), dtype=float),
    ])
    hard_near_train = dist_train < 5000
    hard_near_test = dist_test < 5000

    core_oof_final = np.zeros((len(train_df), 4), dtype=float)
    core_test_final = np.zeros((len(test_df), 4), dtype=float)

    near12_oof = near_lgb_oof.get(24, anchor_oof[12])
    near12_test = near_lgb_test.get(24, anchor_test[12])

    core_oof_final[:, 0] = np.where(
        hard_near_train,
        0.76 * gbsa_core_oof[:, 0] + 0.12 * base_oof["cox"][12] + 0.02 * base_oof["rsf"][12] + 0.10 * near12_oof,
        gbsa_core_oof[:, 0]
    )
    core_test_final[:, 0] = np.where(
        hard_near_test,
        0.76 * gbsa_core_test[:, 0] + 0.12 * base_test["cox"][12] + 0.02 * base_test["rsf"][12] + 0.10 * near12_test,
        gbsa_core_test[:, 0]
    )

    core_oof_final[:, 1] = np.where(
        hard_near_train,
        0.82 * gbsa_core_oof[:, 1] + 0.14 * base_oof["cox"][24] + 0.02 * base_oof["rsf"][24] + 0.02 * near_lgb_oof[24],
        0.62 * gbsa_core_oof[:, 1] + 0.25 * base_oof["cox"][24] + 0.06 * base_oof["rsf"][24] + 0.07 * far_lgb_oof[24]
    )
    core_test_final[:, 1] = np.where(
        hard_near_test,
        0.82 * gbsa_core_test[:, 1] + 0.14 * base_test["cox"][24] + 0.02 * base_test["rsf"][24] + 0.02 * near_lgb_test[24],
        0.62 * gbsa_core_test[:, 1] + 0.25 * base_test["cox"][24] + 0.06 * base_test["rsf"][24] + 0.07 * far_lgb_test[24]
    )

    core_oof_final[:, 2] = np.where(
        hard_near_train,
        0.73 * gbsa_core_oof[:, 2] + 0.16 * base_oof["cox"][48] + 0.03 * base_oof["rsf"][48] + 0.08 * near_lgb_oof[48],
        0.35 * gbsa_core_oof[:, 2] + 0.22 * base_oof["cox"][48] + 0.06 * base_oof["rsf"][48] + 0.37 * far_lgb_oof[48]
    )
    core_test_final[:, 2] = np.where(
        hard_near_test,
        0.73 * gbsa_core_test[:, 2] + 0.16 * base_test["cox"][48] + 0.03 * base_test["rsf"][48] + 0.08 * near_lgb_test[48],
        0.35 * gbsa_core_test[:, 2] + 0.22 * base_test["cox"][48] + 0.06 * base_test["rsf"][48] + 0.37 * far_lgb_test[48]
    )

    core_oof_final[:, 3] = 1.0
    core_test_final[:, 3] = 1.0
    core_oof_final = enforce_monotonicity(np.clip(core_oof_final, 0, 1))
    core_test_final = enforce_monotonicity(np.clip(core_test_final, 0, 1))
else:
    core_oof_final = exp_oof_final.copy()
    core_test_final = exp_test_final.copy()

weights_log["meta_core_mix"] = {"core_weight": 0.70, "exploratory_weight": 0.30}
oof_final = enforce_monotonicity(np.clip(0.70 * core_oof_final + 0.30 * exp_oof_final, 0, 1))
test_final = enforce_monotonicity(np.clip(0.70 * core_test_final + 0.30 * exp_test_final, 0, 1))
oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

if DO_OOF:
    exp_hybrid, exp_c_idx, exp_wb, exp_b24, exp_b48, exp_b72 = compute_hybrid_score(
        time_values, event_values, exp_oof_final[:, 1], exp_oof_final[:, 2], exp_oof_final[:, 3]
    )
    core_hybrid, core_c_idx, core_wb, core_b24, core_b48, core_b72 = compute_hybrid_score(
        time_values, event_values, core_oof_final[:, 1], core_oof_final[:, 2], core_oof_final[:, 3]
    )
    print(f"OOF Exploratory Hybrid = {exp_hybrid:.6f} | C={exp_c_idx:.6f} | WB={exp_wb:.6f}")
    print(f"OOF Core        Hybrid = {core_hybrid:.6f} | C={core_c_idx:.6f} | WB={core_wb:.6f}")

if DO_OOF:
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(
        time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3]
    )
    b12 = compute_brier(time_values, event_values, oof_final[:, 0], 12)
    print(f"OOF Hybrid = {hybrid:.6f}")
    print(f"OOF C-Index = {c_idx:.6f}")
    print(f"OOF WBrier = {wb:.6f}")
    print(f"OOF Brier@12 = {b12:.6f}")
    print(f"OOF Brier@24 = {b24:.6f}")
    print(f"OOF Brier@48 = {b48:.6f}")
    print(f"OOF Brier@72 = {b72:.6f}")

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0) & (vals <= 1)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    oof_dump = pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    })
    oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())




# --- LB-aware embedded consensus + distance calibration ---
core_submission = submission.copy()
core_path = os.path.join(WORK_DIR, "submission_core.csv")
core_submission.to_csv(core_path, index=False)

REQUIRED_COLS = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]

EMBEDDED_SUBMISSIONS = {
    "samplecsv_sub3_0.96961.csv": "event_id,prob_12h,prob_24h,prob_48h,prob_72h\n10662602,0.019747408748853328,0.01974840874885333,0.02318204632059325,1.0\n13353600,0.6571386028177739,0.9214816036146691,0.9631328225485141,1.0\n13942327,0.019548818765518453,0.019549818765518454,0.023114060586979263,1.0\n16112781,0.7009355343741119,0.8840466541804354,0.963398602850256,1.0\n17132808,0.03449737654981883,0.034498376549818834,0.04061907947972944,1.0\n17445696,0.017306112386676238,0.01730711238667624,0.020413951436356437,1.0\n17599982,0.01924992567466107,0.019250925674661072,0.02271653411114625,1.0\n18750374,0.5314428671081441,0.642834215239914,0.832485750808616,1.0\n21365245,0.017156874892996107,0.017157874892996108,0.02227640436501496,1.0\n23634840,0.5260661983551067,0.7747823978176492,0.8814418179995781,1.0\n24461897,0.049075248615224824,0.049076248615224825,0.05171691141414832,1.0\n25353357,0.018408374370839947,0.018409374370839948,0.021647172986467605,1.0\n27052376,0.9348144392412974,0.9348154392412974,0.9404246595984703,1.0\n29312582,0.018399628386363227,0.01840062838636323,0.021547129208015308,1.0\n30451420,0.017421666166570017,0.017422666166570018,0.022750260427843127,1.0\n31488932,0.018474532279623847,0.018475532279623848,0.021731780667767136,1.0\n33717208,0.019484807143500038,0.01948580714350004,0.022982312340623142,1.0\n34826241,0.020556536822740775,0.020557536822740776,0.024066227019091443,1.0\n35311039,0.9826895817618513,0.9896808002490473,0.9923128658967086,1.0\n36423390,0.024026424898748733,0.024027424898748734,0.025325565901006215,1.0\n37196135,0.6608684000710221,0.8686252174871815,0.9558551055482551,1.0\n37355453,0.016283222317859246,0.016284222317859247,0.02128757003479928,1.0\n37536067,0.6499194886559684,0.8842056831178577,0.9589206149919819,1.0\n39717591,0.01770698033545998,0.01770798033545998,0.02085436026161304,1.0\n40833127,0.019498821334078584,0.019499821334078585,0.024530132584955344,1.0\n41907240,0.7332161844181884,0.8996201126354598,0.9704441984719484,1.0\n42963675,0.01786296875188948,0.017863968751889482,0.02305792701976106,1.0\n42997746,0.03457704768943748,0.03457804768943748,0.04086301159373907,1.0\n43341281,0.017879663074013504,0.017880663074013505,0.02306243402598643,1.0\n43371713,0.4620950186947901,0.5919769480234738,0.7705458982403662,1.0\n44872967,0.48454582684263353,0.6718936746473253,0.7817047087102477,1.0\n46574754,0.6066781688767083,0.8179314556903285,0.9010885558932981,1.0\n46767506,0.6081825012126468,0.8451063184215868,0.9408494107716345,1.0\n49169401,0.01630389103977783,0.016304891039777832,0.021396324524374687,1.0\n49502022,0.020108660157823696,0.020109660157823697,0.025264788665214748,1.0\n49565135,0.9724150406554515,0.9902064126281873,0.9902064126281873,1.0\n50233410,0.7315984019326018,0.8836077178282661,0.9620290105278227,1.0\n50541231,0.02937659641935818,0.029377596419358182,0.0356592264650316,1.0\n51664538,0.017518610436379957,0.017519610436379958,0.020678109369700583,1.0\n52004768,0.01810203586175369,0.018103035861753693,0.02311229699645839,1.0\n52039660,0.017931216055415852,0.017932216055415853,0.02111061435723368,1.0\n52614032,0.019008290213437543,0.019009290213437544,0.022362950534031208,1.0\n53407288,0.019849974015329917,0.01985097401532992,0.02318979261533231,1.0\n53799231,0.01997936553631929,0.01998036553631929,0.023095371321592423,1.0\n54348573,0.018799451096858134,0.018800451096858135,0.022057751240183776,1.0\n54459140,0.01428468201329636,0.017507345941896287,0.02292039874398278,1.0\n54759639,0.01929884701250495,0.01929984701250495,0.025052450048374395,1.0\n54801108,0.9182686961933227,0.9653992746929937,0.9895196841397482,1.0\n55504460,0.821723467628627,0.9246546104158778,0.9792688681073235,1.0\n58802003,0.028662926698962185,0.028663926698962186,0.03475098240931602,1.0\n58854430,0.030574784639655527,0.030575784639655528,0.036753332903931665,1.0\n59191113,0.019194682065859683,0.019195682065859684,0.0222845078606516,1.0\n59221605,0.5496955886147104,0.8330176193009646,0.9059578756393428,1.0\n60804856,0.9293119387533982,0.9677707446582764,0.9912201989780929,1.0\n61713947,0.01703330661925588,0.017034306619255882,0.020215257162006596,1.0\n62129708,0.8340026732638063,0.910805497276463,0.9859181877154893,1.0\n66849621,0.9243868477330293,0.9243878477330293,0.9861210097443982,1.0\n67124529,0.017978197093663738,0.01797919709366374,0.02116137293729102,1.0\n69348035,0.01978495091771234,0.01978595091771234,0.023076951653737996,1.0\n69572379,0.01977931037679343,0.019780310376793432,0.02288203273532884,1.0\n69836438,0.016855295514107967,0.01685629551410797,0.02183841263466935,1.0\n70283101,0.6314075982681894,0.729460449001186,0.8924048347563341,1.0\n70501603,0.5158711015706272,0.7735701722716657,0.8700036205745819,1.0\n70993222,0.016905926033751476,0.016906926033751477,0.020053586684608802,1.0\n71474230,0.49963838015750517,0.7533280552478417,0.8679391428579825,1.0\n71532972,0.03222772460020281,0.03222872460020281,0.0383554046449815,1.0\n72973561,0.011628874109959924,0.016833697316065377,0.021995518504752856,1.0\n73023698,0.01862646370070705,0.018627463700707052,0.02215852187527881,1.0\n74751606,0.018294575055849558,0.01829557505584956,0.02176726572917124,1.0\n74945408,0.4319512015516455,0.5474927724146321,0.6761351131294958,1.0\n75012237,0.804138414301931,0.909991178613458,0.9757988540899298,1.0\n75726601,0.01748650289970022,0.017487502899700223,0.022476930615673386,1.0\n78173856,0.006052887700316444,0.015345952299293364,0.02047316276729883,1.0\n78227688,0.033360090010463236,0.03336109001046324,0.0456431211904149,1.0\n78266693,0.01769501273388438,0.01769601273388438,0.021007447314299838,1.0\n78502343,0.019008164226663425,0.019009164226663426,0.022384833945811797,1.0\n78780565,0.01724655430807003,0.01724755430807003,0.0204360041482892,1.0\n80446138,0.02577006487142592,0.02577106487142592,0.031682143802564997,1.0\n80819726,0.00882867051587604,0.01739552827779669,0.022841282820477246,1.0\n81010506,0.017205585057608858,0.01720658505760886,0.020408743930149797,1.0\n81611002,0.017334153851107002,0.017335153851107003,0.022388477615663004,1.0\n81715663,0.5514911112214351,0.7991722241056384,0.8946729337788942,1.0\n82779392,0.017879480359608697,0.017880480359608698,0.02107679440801629,1.0\n83261710,0.017606979493231322,0.017607979493231323,0.022714054382986158,1.0\n84903193,0.020251935911875234,0.020252935911875235,0.023761169664073833,1.0\n87245976,0.019017946290382975,0.019018946290382976,0.022260032321983148,1.0\n87273546,0.04117639054866092,0.04117739054866092,0.047615048790397095,1.0\n89909836,0.018750547455638117,0.018751547455638118,0.023888783307944958,1.0\n91234126,0.04392977391298999,0.04393077391298999,0.05094814529892402,1.0\n93027270,0.622004209015356,0.8665535047977521,0.9383511863948681,1.0\n94627327,0.02946501551016084,0.029466015510160842,0.03574426031750577,1.0\n96570675,0.018750980015890754,0.018751980015890755,0.02373084007408897,1.0\n97225766,0.01890311406159321,0.01890411406159321,0.022154364041814527,1.0\n98446281,0.02917056336855498,0.029171563368554983,0.03529330388832254,1.0\n99649465,0.01946671945520141,0.019467719455201413,0.022841226792782884,1.0\n",
    "samplecsv_sub5_0.97167.csv": "event_id,prob_12h,prob_24h,prob_48h,prob_72h\n10662602,0.014852084557539879,0.024585427284098488,0.02730656144103225,1.0\n13353600,0.7181613033613283,0.931326428235881,0.9632344037321027,1.0\n13942327,0.014718876979365301,0.02429308074550286,0.027238730883015618,1.0\n16112781,0.6942975163325442,0.9225217878138363,0.9601899320476354,1.0\n17132808,0.0189649660864333,0.04784107396226071,0.05212758274619777,1.0\n17445696,0.013283918059286337,0.020780337416510503,0.023275692276318795,1.0\n17599982,0.014303758964721035,0.023878582091564394,0.02669711513569765,1.0\n18750374,0.49795540418344775,0.7434420865911856,0.8420240609288052,1.0\n21365245,0.0135387284142948,0.020486146071065095,0.025364134002886958,1.0\n23634840,0.5267137814971632,0.7851051531794309,0.8779086813741013,1.0\n24461897,0.015663446183581287,0.029342448500233993,0.03412882655973632,1.0\n25353357,0.013626739869999039,0.02257720574309861,0.02513703750882933,1.0\n27052376,0.9373519774045771,0.9563928893839777,0.9563928893839777,1.0\n29312582,0.013992397025481966,0.02246990106039298,0.024899286800437084,1.0\n30451420,0.012821637455690114,0.021008864593003797,0.026149947191745668,1.0\n31488932,0.014676340907555107,0.022476620484957927,0.025100090021415545,1.0\n33717208,0.014230713646784326,0.024337727918510656,0.02716559183982578,1.0\n34826241,0.014765162509341853,0.025981055491909443,0.02871092116491481,1.0\n35311039,0.9895264814060947,0.9941643133504197,0.9941643133504197,1.0\n36423390,0.01593915134419882,0.028701331550070864,0.0316587775445439,1.0\n37196135,0.6709511666953204,0.9106724645845672,0.9518485841413082,1.0\n37355453,0.012660414756744388,0.019264913182973895,0.024088260391058894,1.0\n37536067,0.6967332490781982,0.9191164138305892,0.9542521514656637,1.0\n39717591,0.01356511851363833,0.021416045119656665,0.023937308793999246,1.0\n40833127,0.014136271551584997,0.024278406799073806,0.028733592504064216,1.0\n41907240,0.7679579792368899,0.9528064922470859,0.9709496697547119,1.0\n42963675,0.01280381593338829,0.023150879677136953,0.02833928618206619,1.0\n42997746,0.017932895899041654,0.04821330740436296,0.05270945091349547,1.0\n43341281,0.01344218903645219,0.021735677665800177,0.0266148165629981,1.0\n43371713,0.41083869280101254,0.6552454619406531,0.7846178349751258,1.0\n44872967,0.4434358673216541,0.705077736989648,0.8090799216266831,1.0\n46574754,0.5493695342994596,0.7892955046938105,0.8759710552712758,1.0\n46767506,0.6271556028238996,0.8836627627521161,0.9370278889090329,1.0\n49169401,0.012306826655726537,0.019457495032403946,0.024419246964223904,1.0\n49502022,0.014169307652020383,0.0262374933725315,0.03098334254282996,1.0\n49565135,0.9812057358861075,0.9936591900048031,0.9936591900048031,1.0\n50233410,0.7042786631748141,0.9152321757844761,0.9549762246378418,1.0\n50541231,0.017279728904820785,0.040137755427951415,0.04532197673635376,1.0\n51664538,0.013297290837430876,0.021155369852140476,0.023703372450384508,1.0\n52004768,0.013955074043904479,0.021985051104555682,0.026579880172459998,1.0\n52039660,0.013837862646392932,0.021700919525613822,0.024250219311111515,1.0\n52614032,0.01448548900186955,0.023428533301528536,0.026116941265571004,1.0\n53407288,0.014243558592446158,0.024870743166826505,0.027422421534984212,1.0\n53799231,0.0154975899194403,0.024811507969882376,0.027043165161270515,1.0\n54348573,0.013820383908986033,0.02321958741110737,0.025763882220744286,1.0\n54459140,0.012296908572326182,0.021061056984534247,0.026273827626086727,1.0\n54759639,0.012380946094740646,0.023193530597424974,0.028553046257300457,1.0\n54801108,0.8913858038500933,0.9861592303703747,0.9894052542458647,1.0\n55504460,0.7818627525836137,0.9545578608510065,0.9754933605579791,1.0\n58802003,0.017182803227310934,0.03898918850134662,0.043836771603540305,1.0\n58854430,0.01761349332473739,0.041899449543452746,0.0467926269729068,1.0\n59191113,0.015109506350355541,0.023556931137893217,0.02584302645095993,1.0\n59221605,0.5806795041674282,0.8405189698154382,0.9123736064829935,1.0\n60804856,0.9584532168369976,0.991947774172772,0.991947774172772,1.0\n61713947,0.01302584063194954,0.02041502097876587,0.023061182984857383,1.0\n62129708,0.8546154389740984,0.9777569026830639,0.9867418883105115,1.0\n66849621,0.9242559402566564,0.9871311695114553,0.9900137119349923,1.0\n67124529,0.013643923250996023,0.02186050646499063,0.024392051332214996,1.0\n69348035,0.014422068178203875,0.02466706155529069,0.027162022650563153,1.0\n69572379,0.015357587168878462,0.02451402006294622,0.026750831365066775,1.0\n69836438,0.013084219175739625,0.020094197839302295,0.024792276939119222,1.0\n70283101,0.54615112893319,0.793052226044582,0.8747639158917557,1.0\n70501603,0.5382290520609909,0.7978652533435219,0.8825796722187961,1.0\n70993222,0.013314879961634456,0.020156760764235226,0.022774598054206733,1.0\n71474230,0.483906259925699,0.7418544639315691,0.8489255909846815,1.0\n71532972,0.020237929225965276,0.04407107547807849,0.04864353368547475,1.0\n72973561,0.01224158840146575,0.020100067163404338,0.02503802053236462,1.0\n73023698,0.013958820239702125,0.022948007526922932,0.025937439809979146,1.0\n74751606,0.013841671753814427,0.022394591032167923,0.025338710123505657,1.0\n74945408,0.36474466416779816,0.6205625552264471,0.7245438520442551,1.0\n75012237,0.8325058295104744,0.9716758945342628,0.9838234758493742,1.0\n75726601,0.013775120919180274,0.02099764725363461,0.0256486516728352,1.0\n78173856,0.011473003054373251,0.017887761849482518,0.022969310671357143,1.0\n78227688,0.012676326809758887,0.026911592218803383,0.03315276880421368,1.0\n78266693,0.013586625262076352,0.02142188364406546,0.024194951202770865,1.0\n78502343,0.014712436014650823,0.02337677317748912,0.026110581737888383,1.0\n78780565,0.013291431301067977,0.02069464967287594,0.023336186790344198,1.0\n80446138,0.015407881884863611,0.034661859605965614,0.03974499202412975,1.0\n80819726,0.011817872942811848,0.021507234065274376,0.0269186071057543,1.0\n81010506,0.013338250293918224,0.020679171567235498,0.02333758515345879,1.0\n81611002,0.01355721496888689,0.02077862847057773,0.02553435257243805,1.0\n81715663,0.5307129215538264,0.794397956284068,0.884329931322689,1.0\n82779392,0.01338924696639817,0.021695095181834056,0.024258092049266704,1.0\n83261710,0.013468826625311554,0.021281068907177524,0.026088196524150654,1.0\n84903193,0.014908154285680725,0.025458265369751185,0.02823106048534247,1.0\n87245976,0.014025937216082194,0.02352996458990324,0.026033876454024787,1.0\n87273546,0.020833051281205796,0.05396454370466967,0.05824177548745296,1.0\n89909836,0.014114463088857731,0.02298318726533833,0.0276950397516793,1.0\n91234126,0.024351112050800497,0.05831811157993882,0.06387298195674636,1.0\n93027270,0.6034313055718822,0.8636016652584302,0.9297988287347364,1.0\n94627327,0.01723476374169624,0.040299816672186003,0.045472553487728946,1.0\n96570675,0.014033605334839517,0.023041789760176262,0.027510990167285344,1.0\n97225766,0.013541424931175345,0.023403858098731068,0.025917473241387816,1.0\n98446281,0.017764440283214642,0.039680504567299844,0.04466120429625725,1.0\n99649465,0.01456279383979332,0.02418378504136578,0.02684192351421127,1.0\n",
    "samplecsv_sub6_0.97204.csv": "event_id,prob_12h,prob_24h,prob_48h,prob_72h\n10662602,0.013916194302400023,0.018067462395010205,0.02190617646798639,1.0\n13353600,0.7064502005313547,0.9284448233028054,0.9644302984778975,1.0\n13942327,0.013784785190619661,0.017838797348450647,0.021763280246130542,1.0\n16112781,0.6752156315114148,0.913358832144903,0.9578445146786302,1.0\n17132808,0.018303015704394368,0.035446833223313254,0.0416906061363166,1.0\n17445696,0.012419189987044315,0.015248643874492209,0.01846384853540325,1.0\n17599982,0.013483167948256679,0.01752140308052541,0.021372356697797842,1.0\n18750374,0.48980278046902026,0.7385342723865617,0.8446667885040168,1.0\n21365245,0.01276533053472586,0.015146882480554348,0.020356639314098326,1.0\n23634840,0.5022876803393123,0.7742948277030293,0.8744919809963686,1.0\n24461897,0.017321218290191308,0.02548426096883195,0.035462518438207684,1.0\n25353357,0.012759164541057659,0.016462407354573043,0.019910614107020164,1.0\n27052376,0.8865943545821005,0.9035404295368898,0.9049296375969347,1.0\n29312582,0.013055713727094517,0.016456058085451294,0.01976734740592111,1.0\n30451420,0.013009963783259328,0.015633088511233535,0.02171937397046503,1.0\n31488932,0.013714718119074818,0.016571858839680726,0.020077043741417624,1.0\n33717208,0.013378507165244613,0.01784196643064017,0.0217085709101278,1.0\n34826241,0.013898037583194666,0.018962264908315058,0.022655563690418743,1.0\n35311039,0.9677754936642802,0.9894679486599719,0.991059694605231,1.0\n36423390,0.015544332430646774,0.021997213539509057,0.029331643461868877,1.0\n37196135,0.6341897018266789,0.8920653321276268,0.9440784321060196,1.0\n37355453,0.012005690300164094,0.014479503839485808,0.019706194737942263,1.0\n37536067,0.679457705988263,0.9185816063184323,0.9576689535882772,1.0\n39717591,0.012709949225858578,0.015735263740351956,0.01908902986599766,1.0\n40833127,0.013221859978613049,0.017844782229403965,0.022891232535237827,1.0\n41907240,0.7419418729914706,0.940361216661626,0.9673938945512642,1.0\n42963675,0.012141726879934363,0.018445207002206643,0.0242186515001181,1.0\n42997746,0.018419528164161682,0.03610181962389802,0.043152036216829885,1.0\n43341281,0.012700579068023715,0.016004528772538863,0.021290720943264004,1.0\n43371713,0.39827478794714666,0.6501500720446064,0.7911349781740306,1.0\n44872967,0.4311819053752659,0.691596003740332,0.7869078278550027,1.0\n46574754,0.5127111835163844,0.7963334878026078,0.8827500452367245,1.0\n46767506,0.6069097607380767,0.8752693871252152,0.937626639023094,1.0\n49169401,0.011955010380657834,0.014561935528845554,0.01974991170292654,1.0\n49502022,0.013394215739444323,0.019228353147953934,0.025697900201546994,1.0\n49565135,0.9691615890565534,0.9896537019797363,0.9904282517437417,1.0\n50233410,0.6925394818432946,0.9073170133943999,0.9538989261270734,1.0\n50541231,0.016531922808451335,0.029364675901053985,0.0361268787673372,1.0\n51664538,0.012425568327487417,0.015460867384131959,0.018768932646393996,1.0\n52004768,0.01308759305457273,0.01624722974452346,0.021299651291594415,1.0\n52039660,0.012990412824704624,0.015921208661502966,0.019267080997916578,1.0\n52614032,0.013715290152231874,0.017260799143147085,0.020734573808055143,1.0\n53407288,0.01337156438787924,0.018081539833297838,0.02157428834986782,1.0\n53799231,0.01441937016556249,0.018235733003449925,0.021579056451478257,1.0\n54348573,0.012946187385020364,0.01695524071111114,0.020309713074564466,1.0\n54459140,0.0125580103597724,0.015374077549440743,0.022688736790560066,1.0\n54759639,0.012302968475121759,0.016052787122628812,0.02833716771098669,1.0\n54801108,0.8904250128813535,0.9766262183918306,0.9853088047311332,1.0\n55504460,0.7303882675076332,0.9437318842661112,0.9695277217392791,1.0\n58802003,0.016162899076338395,0.028283229727348237,0.03444497778922206,1.0\n58854430,0.01682028427770306,0.030636299489928055,0.03681110433803028,1.0\n59191113,0.014083913313862877,0.017330733935231386,0.020744059464763844,1.0\n59221605,0.5714169503132496,0.8483808040867684,0.9214402200408327,1.0\n60804856,0.9389022706386474,0.9815297835358314,0.9864069804707287,1.0\n61713947,0.01222313046488362,0.014950234138680615,0.018289834408609547,1.0\n62129708,0.8398845675274556,0.966155844165836,0.9826316559244294,1.0\n66849621,0.9067267124229368,0.9680900065798556,0.9844156156996122,1.0\n67124529,0.012756624047328968,0.01602453268325958,0.019334818458366144,1.0\n69348035,0.013480255872772826,0.018047711679599853,0.021445483279767377,1.0\n69572379,0.01430631101990867,0.01802284194375734,0.021346540585100465,1.0\n69836438,0.012334486418856549,0.014920089209838103,0.02001051553104197,1.0\n70283101,0.5357057565027787,0.7862982710111243,0.8680976189597981,1.0\n70501603,0.5136562362953487,0.7876359364124695,0.8811105292562921,1.0\n70993222,0.012418999798626188,0.014761523379099748,0.01808491625519699,1.0\n71474230,0.45894805615795864,0.7244086347194166,0.8377228230581263,1.0\n71532972,0.01921237002476433,0.03228780772245008,0.03876006993923604,1.0\n72973561,0.01193268522994743,0.014926355377443908,0.02020315430020817,1.0\n73023698,0.013106743352373055,0.016873468721443872,0.02095944741668641,1.0\n74751606,0.013058922061406172,0.01643256854266655,0.020085929432732255,1.0\n74945408,0.37281628437504566,0.6219253824848131,0.7423623387862709,1.0\n75012237,0.8248044101087302,0.9665819650122143,0.9811720970530591,1.0\n75726601,0.012973585055593092,0.015512090897462937,0.0205822623800497,1.0\n78173856,0.01097670302344994,0.01334962334800442,0.01862491464961952,1.0\n78227688,0.015421102461544548,0.02181868075668609,0.03540869705636562,1.0\n78266693,0.012742620529043944,0.015651315778993882,0.019163443131883864,1.0\n78502343,0.013750795923103359,0.017179816187313974,0.020823658196109155,1.0\n78780565,0.01240491840678226,0.015175815322496454,0.018521982332147142,1.0\n80446138,0.014821394300016826,0.02599073000693936,0.0322358754325905,1.0\n80819726,0.011460971580192398,0.018039705703249743,0.02402140414851622,1.0\n81010506,0.01263204931500852,0.015144755275244368,0.018542675761177893,1.0\n81611002,0.01272545537016243,0.015438909355140973,0.020480705461257936,1.0\n81715663,0.5052327994060114,0.7672301143320562,0.8628210563665913,1.0\n82779392,0.012530078993037251,0.01588321178180437,0.0192141234743403,1.0\n83261710,0.012657076800741731,0.015676795193420495,0.020880489048635365,1.0\n84903193,0.014031124574902226,0.01861655694615888,0.0223904179484115,1.0\n87245976,0.013091652492460354,0.017186690272390556,0.020575752213941454,1.0\n87273546,0.024336941147706334,0.04048180125395317,0.04744926905199284,1.0\n89909836,0.013252075968401549,0.016942407510303247,0.022145898941763092,1.0\n91234126,0.023300585900424883,0.04191814453804701,0.04850969900476569,1.0\n93027270,0.5684484137256481,0.8304694103012408,0.9058811234055261,1.0\n94627327,0.016388545255944065,0.029541343410988876,0.036190043077230186,1.0\n96570675,0.0131296383902482,0.016977651758590707,0.02202831771870783,1.0\n97225766,0.012686823463411554,0.017064816770661614,0.020432883206060515,1.0\n98446281,0.016864385724832474,0.02923876319771143,0.035513282755813357,1.0\n99649465,0.013613070776021668,0.017713998044287665,0.021259371596427837,1.0\n",
    "samplecsv_sub8_0.97058.csv": "event_id,prob_12h,prob_24h,prob_48h,prob_72h\n10662602,0.01400566004626915,0.01960477352759859,0.02450262006065118,1.0\n13353600,0.7043944312899552,0.9244133361871204,0.9583808749979972,1.0\n13942327,0.013886275787728473,0.01936081048445575,0.024370993735185992,1.0\n16112781,0.6968198254931572,0.9140397032401244,0.9536286588642865,1.0\n17132808,0.018200903060209626,0.038740230594036786,0.045914358540725696,1.0\n17445696,0.012511172051678763,0.016571197721588563,0.0209126622524096,1.0\n17599982,0.013551417371525603,0.019046242130043932,0.02395598983556573,1.0\n18750374,0.49844963701462275,0.737360748769434,0.8453818153342907,1.0\n21365245,0.012839953059532394,0.01632526452568075,0.022766658900142598,1.0\n23634840,0.5185137419983842,0.7957151362661709,0.8828153104900257,1.0\n24461897,0.02388331037480327,0.04144886495253671,0.05224546438816754,1.0\n25353357,0.012844327754603084,0.017934374249907556,0.02242021126664787,1.0\n27052376,0.8978370474103017,0.9147041338477466,0.9147041338477466,1.0\n29312582,0.01315758555829568,0.017894703787482185,0.022308341362894803,1.0\n30451420,0.012871356196034108,0.01683807499622463,0.023969251044328,1.0\n31488932,0.013828045348515342,0.017969874449890975,0.02263110250472179,1.0\n33717208,0.013454461012406826,0.019432867788297375,0.024353801429870887,1.0\n34826241,0.013971569525894687,0.020660373013950503,0.02543419911804887,1.0\n35311039,0.9687515824467869,0.9891955125390548,0.9894732938704873,1.0\n36423390,0.015353174446723995,0.02427877659924383,0.032910707903345326,1.0\n37196135,0.6128860477032739,0.8909156657699298,0.9429728967123033,1.0\n37355453,0.012079771402920162,0.015631417401225093,0.022038791887025913,1.0\n37536067,0.6799031896464114,0.9217545496755427,0.9565727214564301,1.0\n39717591,0.012805243075893685,0.017087707042701933,0.021503673174233053,1.0\n40833127,0.013318695569327578,0.019408533564114556,0.02562830820753948,1.0\n41907240,0.7365615467231612,0.9305629775631421,0.9610801383142904,1.0\n42963675,0.012217896860367713,0.02003698945452138,0.026980437404287443,1.0\n42997746,0.018101785899493617,0.03940454019566237,0.04714070971583253,1.0\n43341281,0.012772204376568275,0.01736207666975173,0.023844223843957776,1.0\n43371713,0.40588755403648913,0.6734068638786607,0.8028868884286012,1.0\n44872967,0.42861544837507093,0.704227485066916,0.7933349692080359,1.0\n46574754,0.5095744690831583,0.7766944204723961,0.8651834099489982,1.0\n46767506,0.5888535241709894,0.8746803127248453,0.9367178143828729,1.0\n49169401,0.011962738619650056,0.015738621715584877,0.02211409629801216,1.0\n49502022,0.013426369114262681,0.020755001246493734,0.02821027489850617,1.0\n49565135,0.9684811388506432,0.9897229338744867,0.9897229338744867,1.0\n50233410,0.6963664436371627,0.8934014036243169,0.9405835864760409,1.0\n50541231,0.016512550937427716,0.03207214413092749,0.03979247435028703,1.0\n51664538,0.012521734583865345,0.01683694564619053,0.021222166717343574,1.0\n52004768,0.013173029453715391,0.017589665932395082,0.023868126788915683,1.0\n52039660,0.013083333654189356,0.017290022420865973,0.02175903938226792,1.0\n52614032,0.013789441199402454,0.01874283808223195,0.0234128239516278,1.0\n53407288,0.013454165715894502,0.019723108889871378,0.024267784438564748,1.0\n53799231,0.014544406275303505,0.019819416534364632,0.024364461487981257,1.0\n54348573,0.013034245974371285,0.018502406269665646,0.02296268164344278,1.0\n54459140,0.012409351366032611,0.01655286210022228,0.024536001667724714,1.0\n54759639,0.012240390534483196,0.017413401394209566,0.02904104094466926,1.0\n54801108,0.8838813903737762,0.9609113733117001,0.9726423073632944,1.0\n55504460,0.7217271573952904,0.9205024124887528,0.9519788567861008,1.0\n58802003,0.016194087775009317,0.03102741072708822,0.03826807025371072,1.0\n58854430,0.016817666003802016,0.03352988959667283,0.04075956826931673,1.0\n59191113,0.014204539456368945,0.01881979925764901,0.023339160929422245,1.0\n59221605,0.5814044974691396,0.8622038219439988,0.9254469637323086,1.0\n60804856,0.9389220892547911,0.9768057815519153,0.9816256232442662,1.0\n61713947,0.012306951884356918,0.016272517762130227,0.02069389741088567,1.0\n62129708,0.8490977230262379,0.962160244204516,0.9799959667678306,1.0\n66849621,0.904227358418108,0.9563157333566288,0.9794832340282196,1.0\n67124529,0.012851014134626838,0.017434303029504834,0.0218627104562194,1.0\n69348035,0.01358756320534964,0.019721411522673107,0.024257577580588097,1.0\n69572379,0.014424667193239843,0.019576091673908006,0.024097141073914304,1.0\n69836438,0.01240748927002162,0.016142647835377884,0.022427460800588935,1.0\n70283101,0.5493548192359229,0.7991496089376233,0.8763062466871429,1.0\n70501603,0.5062768169203679,0.7926202321970249,0.8890891017484417,1.0\n70993222,0.012521767368097179,0.016027616861216072,0.02045309136712625,1.0\n71474230,0.47750911663391843,0.7455116078032555,0.8442715464915885,1.0\n71532972,0.0192111578943012,0.035232084602516485,0.0427976131963084,1.0\n72973561,0.011925494186320794,0.016164811147497325,0.02258347466251887,1.0\n73023698,0.01319191410775118,0.018341907018796382,0.023450074595296785,1.0\n74751606,0.01313945064898254,0.017836803509964274,0.022602000136035152,1.0\n74945408,0.37341475571149246,0.621249597312522,0.7432004456231082,1.0\n75012237,0.840034029851182,0.9697960957931574,0.9810324575837188,1.0\n75726601,0.013043813993387228,0.016743111831859357,0.023033926205056544,1.0\n78173856,0.011023727493758042,0.014458233177208783,0.02088601779534572,1.0\n78227688,0.023556908019815283,0.0366972832377738,0.05072349965234267,1.0\n78266693,0.012835046698748999,0.01701508471098925,0.021619223691917818,1.0\n78502343,0.01386132299040867,0.01865202521736587,0.02341603578554851,1.0\n78780565,0.01250843709709583,0.016499849435776016,0.020943098411517532,1.0\n80446138,0.014807812043031516,0.02837789734491802,0.03565021077924056,1.0\n80819726,0.011472779805751185,0.019570235434052074,0.02669228675809179,1.0\n81010506,0.012686422504171182,0.016446161252755415,0.020932293414367534,1.0\n81611002,0.012810361628756829,0.016653693713444107,0.022957277230475037,1.0\n81715663,0.5120442604943657,0.7801681206898468,0.865026895944158,1.0\n82779392,0.012618779162223858,0.017288439576674572,0.021691879398375358,1.0\n83261710,0.012742789892845075,0.016999892008377288,0.02341309916628197,1.0\n84903193,0.014105338771174918,0.020233597736577697,0.025085945456040405,1.0\n87245976,0.013199252494460103,0.01874599964284516,0.023221973950975056,1.0\n87273546,0.023218768151935153,0.04380669954774346,0.05156291581204335,1.0\n89909836,0.013334143933446303,0.018316213593793058,0.024718208645983782,1.0\n91234126,0.023144024045238458,0.045401606429518175,0.05292026061925028,1.0\n93027270,0.577074953227291,0.8343907037622288,0.9031848317116188,1.0\n94627327,0.016408473532801057,0.03227089069005748,0.039905863347363554,1.0\n96570675,0.01322319255930834,0.018426701942732212,0.02465341420458009,1.0\n97225766,0.012778938491401402,0.018646705224799342,0.023064064425671986,1.0\n98446281,0.0168802094016481,0.03188343132165028,0.0392402979876816,1.0\n99649465,0.0137092857242954,0.019273826890947265,0.02392375005216782,1.0\n",
}

def _rank_to_core_distribution(core_vals, rank_signal):
    order = np.argsort(rank_signal, kind="mergesort")
    sorted_core = np.sort(core_vals)
    out = np.empty_like(sorted_core)
    out[order] = sorted_core
    return out

def _safe_align_submission_df(df, sample_ids):
    if list(df.columns) != REQUIRED_COLS:
        return None
    if len(df) != len(sample_ids):
        return None
    if df["event_id"].duplicated().any():
        return None
    if not df["event_id"].equals(sample_ids):
        try:
            df = sample_ids.to_frame(name="event_id").merge(df, on="event_id", how="left", validate="one_to_one")
        except Exception:
            return None
        if df.isnull().any().any():
            return None
    vals = df[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        return None
    vals = np.clip(vals, 0.0, 1.0)
    vals = enforce_monotonicity(vals)
    vals[:, 3] = 1.0
    df = df.copy()
    df.loc[:, REQUIRED_COLS[1:]] = vals
    return df

def _safe_read_submission_csv(path, sample_ids):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    return _safe_align_submission_df(df, sample_ids)

def _safe_read_submission_text(text, sample_ids):
    try:
        df = pd.read_csv(StringIO(text))
    except Exception:
        return None
    return _safe_align_submission_df(df, sample_ids)

def _extract_score_from_name(path):
    name = os.path.basename(path)
    m = re.findall(r"(\d\.\d{4,5})", name)
    if not m:
        return None
    try:
        return max(float(x) for x in m)
    except Exception:
        return None

def _hash_frame(df):
    arr = np.round(df[REQUIRED_COLS[1:]].to_numpy(dtype=float), 8)
    return hashlib.md5(arr.tobytes()).hexdigest()

def _weighted_candidate_mean(candidate_frames, candidate_meta, core_submission):
    core_arr = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    mad_vals = np.array([max(meta["mad_to_core_12_48"], 1e-9) for meta in candidate_meta], dtype=float)
    mad_ref = float(np.median(mad_vals)) if mad_vals.size else 1.0
    mad_ref = max(mad_ref, 1e-6)

    score_vals = np.array([meta["score"] if meta["score"] is not None else np.nan for meta in candidate_meta], dtype=float)
    finite_scores = score_vals[np.isfinite(score_vals)]
    best_score = float(np.max(finite_scores)) if finite_scores.size else 0.9725

    raw_weights = []
    for meta in candidate_meta:
        s = meta["score"] if meta["score"] is not None else (best_score - 0.0015)
        score_term = math.exp((s - best_score) / 0.00080)
        novelty_term = max(0.50, meta["mad_to_core_12_48"] / mad_ref) ** 0.35
        corr_term = max(0.85, meta["mean_corr_12_48"]) ** 0.10
        raw_weights.append(score_term * novelty_term * corr_term)

    raw_weights = np.asarray(raw_weights, dtype=float)
    raw_weights = np.where(np.isfinite(raw_weights), raw_weights, 0.0)
    if raw_weights.sum() <= 0:
        raw_weights = np.ones(len(candidate_meta), dtype=float)
    weights = raw_weights / raw_weights.sum()

    candidate_arrays = np.stack([df[REQUIRED_COLS[1:]].to_numpy(dtype=float) for df in candidate_frames], axis=0)
    prob_mean = np.tensordot(weights, candidate_arrays, axes=(0, 0))

    rankblend = np.zeros_like(prob_mean)
    for j in range(4):
        rank_signal = np.tensordot(
            weights,
            np.stack([df[REQUIRED_COLS[1:]].rank(method="average", pct=True).to_numpy(dtype=float)[:, j] for df in candidate_frames], axis=0),
            axes=(0, 0)
        )
        rankblend[:, j] = _rank_to_core_distribution(core_arr[:, j], rank_signal)

    return weights, prob_mean, rankblend

def _fit_distance_bin_prior(train_df, test_df, horizon, strength=0.30):
    bins = np.array([0, 1000, 2000, 3000, 4000, 5000, 6000, 8000, 10000, 15000, np.inf], dtype=float)
    y, valid = make_binary_target(train_df["time_to_hit_hours"].values, train_df["event"].values, horizon)
    tmp = train_df.loc[valid, ["dist_min_ci_0_5h"]].copy()
    tmp["y"] = y[valid]
    tmp["bin"] = pd.cut(tmp["dist_min_ci_0_5h"], bins=bins, labels=False, include_lowest=True, right=False)
    global_mean = float(tmp["y"].mean()) if len(tmp) else 0.0
    stats = tmp.groupby("bin")["y"].agg(["sum", "count"]) if len(tmp) else pd.DataFrame(columns=["sum", "count"])

    prob_map = {}
    for b in range(len(bins) - 1):
        if b in stats.index:
            s = float(stats.loc[b, "sum"])
            c = float(stats.loc[b, "count"])
            prob_map[b] = (s + strength * global_mean) / (c + strength)
        else:
            prob_map[b] = global_mean

    test_bins = pd.cut(test_df["dist_min_ci_0_5h"], bins=bins, labels=False, include_lowest=True, right=False)
    out = np.array([
        prob_map.get(int(b), global_mean) if pd.notna(b) else global_mean
        for b in test_bins
    ], dtype=float)
    return np.clip(out, 0.0, 1.0)

def find_candidate_submission_paths():
    roots = []
    if os.path.isdir("/kaggle/input"):
        roots.append("/kaggle/input")
    if os.path.isdir("/kaggle/working"):
        roots.append("/kaggle/working")
    roots.append(".")
    deny_names = {
        "train.csv", "test.csv", "sample_submission.csv",
        "submission.csv", "submission_core.csv", "submission_consensus.csv",
        "submission_external.csv", "oof_predictions.csv"
    }
    paths = []
    seen = set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cur, _, files in os.walk(root):
            for fn in files:
                if not fn.lower().endswith(".csv"):
                    continue
                if fn in deny_names:
                    continue
                full = os.path.join(cur, fn)
                if full in seen:
                    continue
                seen.add(full)
                paths.append(full)
    return sorted(paths)

sample_ids = sample_sub["event_id"]
candidate_frames = []
candidate_meta = []
core_arr = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)

# embedded candidates
for name, text in EMBEDDED_SUBMISSIONS.items():
    cand = _safe_read_submission_text(text, sample_ids)
    if cand is None:
        continue
    if np.allclose(cand[REQUIRED_COLS[1:]].to_numpy(), core_arr, atol=1e-12):
        continue
    score = _extract_score_from_name(name)
    if score is not None and score < 0.9690:
        continue
    arr = cand[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    corrs = []
    for j in range(3):
        x = arr[:, j]
        y = core_arr[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corrs.append(1.0)
        else:
            corrs.append(float(np.corrcoef(x, y)[0, 1]))
    mean_corr = float(np.mean(corrs))
    if not np.isfinite(mean_corr) or mean_corr < 0.80:
        continue
    candidate_frames.append(cand)
    candidate_meta.append({
        "path": f"embedded::{name}",
        "score": score,
        "mean_corr_12_48": mean_corr,
        "mad_to_core_12_48": float(np.mean(np.abs(arr[:, :3] - core_arr[:, :3]))),
        "hash": _hash_frame(cand),
        "source": "embedded",
    })

# optional extra CSV candidates from working/input
for path in find_candidate_submission_paths():
    cand = _safe_read_submission_csv(path, sample_ids)
    if cand is None:
        continue
    arr = cand[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if np.allclose(arr, core_arr, atol=1e-12):
        continue
    score = _extract_score_from_name(path)
    if score is not None and score < 0.9690:
        continue
    corrs = []
    for j in range(3):
        x = arr[:, j]
        y = core_arr[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corrs.append(1.0)
        else:
            corrs.append(float(np.corrcoef(x, y)[0, 1]))
    mean_corr = float(np.mean(corrs))
    if not np.isfinite(mean_corr) or mean_corr < 0.80:
        continue
    candidate_frames.append(cand)
    candidate_meta.append({
        "path": path,
        "score": score,
        "mean_corr_12_48": mean_corr,
        "mad_to_core_12_48": float(np.mean(np.abs(arr[:, :3] - core_arr[:, :3]))),
        "hash": _hash_frame(cand),
        "source": "file",
    })

# numeric dedup
dedup_frames = []
dedup_meta = []
seen_hash = set()
for df, meta in zip(candidate_frames, candidate_meta):
    if meta["hash"] in seen_hash:
        continue
    seen_hash.add(meta["hash"])
    dedup_frames.append(df)
    dedup_meta.append(meta)

candidate_frames = dedup_frames
candidate_meta = dedup_meta

consensus_submission = core_submission.copy()
consensus_path = os.path.join(WORK_DIR, "submission_consensus.csv")

if candidate_frames:
    weights, prob_mean, rankblend = _weighted_candidate_mean(candidate_frames, candidate_meta, core_submission)

    core_vals = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    final_vals = core_vals.copy()

    prob_w = np.array([0.015, 0.045, 0.110, 0.000], dtype=float)
    rank_w = np.array([0.015, 0.035, 0.055, 0.000], dtype=float)

    for j in range(4):
        keep_w = 1.0 - prob_w[j] - rank_w[j]
        final_vals[:, j] = keep_w * core_vals[:, j] + prob_w[j] * prob_mean[:, j] + rank_w[j] * rankblend[:, j]

    final_vals = np.clip(final_vals, 0.0, 1.0)
    final_vals = enforce_monotonicity(final_vals)
    final_vals[:, 3] = 1.0
    consensus_submission.loc[:, REQUIRED_COLS[1:]] = final_vals
else:
    weights = np.array([], dtype=float)
    prob_w = np.array([0.0, 0.0, 0.0, 0.0], dtype=float)
    rank_w = np.array([0.0, 0.0, 0.0, 0.0], dtype=float)

validate_submission(consensus_submission, sample_sub)
consensus_submission.to_csv(consensus_path, index=False)

# distance-aware final calibration
dist_prior_test = np.column_stack([
    _fit_distance_bin_prior(train_df, test_df, 12, strength=0.30),
    _fit_distance_bin_prior(train_df, test_df, 24, strength=0.30),
    _fit_distance_bin_prior(train_df, test_df, 48, strength=0.30),
    np.ones(len(test_df), dtype=float),
])

dist_vals = np.asarray(test_df["dist_min_ci_0_5h"], dtype=float)
far_gate = 1.0 / (1.0 + np.exp(-(dist_vals - 5200.0) / 700.0))

alpha_near = np.array([0.06, 0.10, 0.14], dtype=float)
alpha_far = np.array([0.18, 0.28, 0.38], dtype=float)

final_vals = consensus_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float).copy()
for j in range(3):
    alpha = alpha_near[j] * (1.0 - far_gate) + alpha_far[j] * far_gate
    final_vals[:, j] = (1.0 - alpha) * final_vals[:, j] + alpha * dist_prior_test[:, j]

final_vals = np.clip(final_vals, 0.0, 1.0)
final_vals = enforce_monotonicity(final_vals)
final_vals[:, 3] = 1.0

external_submission = core_submission.copy()
external_submission.loc[:, REQUIRED_COLS[1:]] = final_vals
validate_submission(external_submission, sample_sub)

external_path = os.path.join(WORK_DIR, "submission_external.csv")
external_submission.to_csv(external_path, index=False)
external_submission.to_csv(OUTPUT_PATH, index=False)

with open(os.path.join(WORK_DIR, "external_blend_summary.json"), "w") as f:
    json.dump({
        "n_candidates": len(candidate_meta),
        "weights": [
            {
                "path": m["path"],
                "source": m["source"],
                "score": m["score"],
                "mean_corr_12_48": m["mean_corr_12_48"],
                "mad_to_core_12_48": m["mad_to_core_12_48"],
                "weight": float(w),
            }
            for m, w in zip(candidate_meta, weights)
        ],
        "consensus_blend": {
            "prob_w": prob_w.tolist(),
            "rank_w": rank_w.tolist(),
        },
        "distance_prior": {
            "strength": 0.30,
            "bins_m": [0, 1000, 2000, 3000, 4000, 5000, 6000, 8000, 10000, 15000, "inf"],
            "alpha_near": alpha_near.tolist(),
            "alpha_far": alpha_far.tolist(),
            "soft_gate_center_m": 5200.0,
            "soft_gate_width_m": 700.0,
        }
    }, f, indent=2)

print("Embedded/file candidates used:", len(candidate_meta))
for m, w in sorted(zip(candidate_meta, weights), key=lambda z: -z[1])[:20]:
    print(f"  w={w:.4f} | score={m['score']} | corr={m['mean_corr_12_48']:.4f} | {m['path']}")

print("Saved:", core_path)
print("Saved:", consensus_path)
print("Saved:", external_path)
print("Saved:", OUTPUT_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 16.4 MB/s eta 0:00:00
DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35) HAS_SKSURV = True
anchor@12  Brier = 0.068845
anchor@24  Brier = 0.029693
anchor@48  Brier = 0.019065
global_lgb@12  Brier = 0.059245
global_cat@12  Brier = 0.055916
linear@12      Brier = 0.057712
global_lgb@24  Brier = 0.027575
global_cat@24  Brier = 0.026686
linear@24      Brier = 0.029833
global_lgb@48  Brier = 0.015347
global_cat@48  Brier = 0.015392
linear@48      Brier = 0.017817
near_lgb@12    Brier = 0.056701
near_lgb@24    Brier = 0.027141
near_lgb@48    Brier = 0.014885
far_lgb@24     Brier = 0.027604
far_lgb@48     Brier = 0.014642
timing@12      Brier = 0.211033
timing@24      Brier = 0.340257
timing@48      Brier = 0.368051
gbsa@12        Br